# Предсказание ориентации текста по готовым весам

Блокнот загружает обученную CNN и создаёт `submission.csv`: вероятность `p_180` того, что изображение перевёрнуто на 180°. Обучение и загрузка сторонних датасетов не требуются.

Установите зависимости из `req.txt`, выберите ядро этого окружения и выполните **Restart Kernel → Run All** из корня проекта. Нужны готовые веса, тестовые изображения и `sample_submission.csv` по путям ниже.

Архитектура и предобработка совпадают с `main.ipynb`: RGB → три одинаковых канала серого → размер 384 × 64 → значения в [0, 1]. Используется один проход без TTA. Идентификаторы и порядок строк берутся из образца. Результат записывается в `submission.csv` в корне проекта; повторный запуск перезаписывает этот файл.

Вычисления в приложенном файле проводились на mps.

In [2]:
import csv
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

CHECKPOINT_PATH = Path("artifacts/best_orientation_model_7conv.pt")
TEST_IMAGES_DIR = Path("test/test/images")
SAMPLE_SUBMISSION_PATH = Path("test/sample_submission.csv")
SUBMISSION_PATH = Path("submission.csv")
IMAGE_WIDTH, IMAGE_HEIGHT = 384, 64
BATCH_SIZE = 128
EXPECTED_IMAGES = 20_000

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Устройство: {device}")

Устройство: mps


## Архитектура и загрузка весов

Имя checkpoint исторически содержит `7conv`; модель имеет шесть свёрточных слоёв. `eval()` отключает Dropout и переводит BatchNorm в режим предсказания.

In [4]:
class TinyOrientationNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.10),
            
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        
        self.classifier = nn.Sequential(nn.Dropout(0.30), nn.Linear(128, 2))

    def forward(self, images):
        return self.classifier(self.features(images).flatten(1))

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Не найдены веса: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = TinyOrientationNet()
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
# Отключаем Dropout и используем накопленные статистики BatchNorm.
model = model.to(device).eval()
print(f"Загружены веса: {CHECKPOINT_PATH}")
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

Загружены веса: artifacts/best_orientation_model_7conv.pt
Параметров: 393,858


## Тестовые изображения

Сопоставляем `image_id` из образца с именами файлов без расширений. Проверяем количество, уникальность и наличие изображений до запуска модели.

In [ ]:
with SAMPLE_SUBMISSION_PATH.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    if reader.fieldnames != ["image_id", "p_180"]:
        raise ValueError("Ожидаются колонки image_id,p_180 в sample_submission.csv")
    image_ids = [row["image_id"] for row in reader]

if len(image_ids) != EXPECTED_IMAGES or len(set(image_ids)) != EXPECTED_IMAGES:
    raise ValueError(f"Образец должен содержать {EXPECTED_IMAGES} уникальных ID")
if not TEST_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Не найдена папка: {TEST_IMAGES_DIR}")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
paths_by_id = {}
for path in sorted(TEST_IMAGES_DIR.iterdir()):
    if path.is_file() and path.suffix.lower() in extensions:
        if path.stem in paths_by_id:
            raise ValueError(f"Несколько изображений с ID {path.stem}")
        paths_by_id[path.stem] = path
missing = set(image_ids) - set(paths_by_id)
if missing:
    raise FileNotFoundError(f"Нет изображений для {len(missing)} ID: {sorted(missing)[:5]}")
# Сохраняем порядок изображений из образца submission,
# чтобы каждая вероятность соответствовала нужному image_id.
image_paths = [paths_by_id[image_id] for image_id in image_ids]
print(f"Подготовлено изображений: {len(image_paths)}")

Подготовлено изображений: 20000


In [7]:
val_transforms = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH), interpolation=Image.Resampling.BILINEAR),
])

class TestImageDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = val_transforms(image.convert("RGB"))
        pixels = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(pixels).permute(2, 0, 1)

test_loader = DataLoader(
    TestImageDataset(image_paths), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=0,
)

## Предсказания и сохранение

Берём вероятность класса 1 после softmax. Перед записью проверяем количество и диапазон вероятностей; ID сохраняем ровно как в образце, без расширений файлов.

In [ ]:
probability_batches = []
# При предсказании градиенты не нужны — это уменьшает расход памяти.
with torch.no_grad():
    for batch_index, images in enumerate(test_loader, start=1):
        logits = model(images.to(device))
        probability_batches.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        if batch_index % 25 == 0 or batch_index == len(test_loader):
            print(f"Обработано батчей: {batch_index}/{len(test_loader)}")
probabilities = np.concatenate(probability_batches)
if len(probabilities) != len(image_ids):
    raise ValueError("Число предсказаний не совпадает с числом ID")
if not np.all(np.isfinite(probabilities) & (probabilities >= 0) & (probabilities <= 1)):
    raise ValueError("Вероятности должны быть конечными и находиться в [0, 1]")

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
with SUBMISSION_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["image_id", "p_180"])
    writer.writerows((image_id, f"{probability:.8f}")
                     for image_id, probability in zip(image_ids, probabilities))
print(f"Сохранено {len(probabilities)} предсказаний: {SUBMISSION_PATH.resolve()}")

Обработано батчей: 25/157
Обработано батчей: 50/157
Обработано батчей: 75/157
Обработано батчей: 100/157
Обработано батчей: 125/157
Обработано батчей: 150/157
Обработано батчей: 157/157
Сохранено 20000 предсказаний: /Users/atweshka/detect_orientation/submission.csv
